## Paso 1 — Validez: reconfirmar hallazgos conocidos + rangos físicos (§5.3 del plan)

Completitud y unicidad ya quedaron cubiertas en los notebooks 01 y 02 — no
se repiten aquí. Este paso reconfirma con código propio 2 hallazgos ya
documentados en `reporte-inconsistencias.md` (HU-10.3): una solicitud con
`Deadline` anterior a su propia creación, y una ruta malformada
(`"b"`/`"j"`, ya conocida como dato de prueba). También revisamos que los
valores estén en rangos físicamente posibles (`ProbabilidadCierre`/
`ScorePrioridad` entre 0-100, montos no negativos).


In [1]:
import pandas as pd
from pathlib import Path

RUTA_INTERIM = Path("../datos/interim")
ARCHIVOS_PARQUET = sorted(RUTA_INTERIM.glob("*.parquet"))
tablas = {archivo.stem: pd.read_parquet(archivo) for archivo in ARCHIVOS_PARQUET}
assert len(tablas) == 18, f"Se esperaban 18 tablas, se cargaron {len(tablas)}"

solicitudes = tablas["solicitudes"]
catalogo_rutas = tablas["catalogo_rutas"]
lineas_solicitud = tablas["lineas_solicitud"]

# --- 1. Deadline anterior a CreatedAt ---
con_deadline = solicitudes.dropna(subset=["deadline"])
deadline_invalido = con_deadline[con_deadline["deadline"] < con_deadline["createdAt"]]
print(f"Solicitudes con Deadline < CreatedAt: {len(deadline_invalido)}")
print(deadline_invalido[["id", "deadline", "createdAt"]].to_string())

# --- 2. Ruta malformada ("b"/"j", ya conocida) ---
rutas_malformadas = catalogo_rutas[catalogo_rutas["origen"].isin(["b", "j", "222", "tr"])]
print(f"\nRutas malformadas encontradas: {len(rutas_malformadas)}")
print(rutas_malformadas[["id", "origen", "destino"]].to_string())

# --- 3. Rangos físicos ---
print("\n--- Rangos físicos ---")
fuera_de_rango_prob = solicitudes[(solicitudes["probabilidadCierre"] < 0) | (solicitudes["probabilidadCierre"] > 100)]
print(f"ProbabilidadCierre fuera de 0-100: {len(fuera_de_rango_prob)}")

fuera_de_rango_score = solicitudes[(solicitudes["scorePrioridad"] < 0) | (solicitudes["scorePrioridad"] > 100)]
print(f"ScorePrioridad fuera de 0-100: {len(fuera_de_rango_score)}")

columnas_monetarias = {
    "solicitudes": ["valorEstimado"],
    "cotizacion_lineas": ["valorFlete", "valorAprovisionamiento", "valorRetornoContenedor"],
    "lineas_solicitud": ["valorAprovisionamiento", "valorMercancia"],
    "tarifas_historicas": ["valor"],
}
for nombre_tabla, columnas in columnas_monetarias.items():
    df = tablas[nombre_tabla]
    for col in columnas:
        negativos = df[df[col] < 0]
        if len(negativos) > 0:
            print(f"{nombre_tabla}.{col}: {len(negativos)} valores negativos")
print("(sin más líneas arriba = ningún monto negativo encontrado)")


Solicitudes con Deadline < CreatedAt: 1
                                      id            deadline           createdAt
12  3EAB0AB6-ED6D-4983-A8EA-F9BF7406AB0A 2026-08-26 16:44:00 2026-08-31 16:44:58

Rutas malformadas encontradas: 3
                                      id origen destino
5   7309C6DC-EEC4-4EB8-9FBC-24D0D5BAB29C    222     222
6   0571BA81-5C4D-4325-A957-30163A4F6BBB      b       j
24  D7BAC012-7CAE-419F-9A1E-C401C01CA58C     tr      rr

--- Rangos físicos ---
ProbabilidadCierre fuera de 0-100: 0
ScorePrioridad fuera de 0-100: 0
(sin más líneas arriba = ningún monto negativo encontrado)


## Paso 2 — Reconfirmar Fuente=Sicetac/Tarifario contra la fuente externa (§5.4 del plan)

El hallazgo más importante de HU-10.3: de las líneas de cotización con
`Fuente` en `Sicetac`/`Tarifario`, ¿el valor cotizado existe de verdad en
la fuente externa correspondiente? Metodología idéntica al reporte
original: buscar el valor exacto de `ValorFlete` dentro de
`VALORSICECONHORA` (SICE-TAC, núcleo+raras) para `Fuente=Sicetac`, o
dentro de `VALOR` (tarifa-plaza) para `Fuente=Tarifario`. Se espera
14/15 — la única falla conocida es la ruta malformada "b"/"j"
(`LineaCotizacionId CA0FC8A8-...`), que no se puede verificar por esa
razón, no porque el motor de costeo falle.

In [2]:
cotizacion_lineas = tablas["cotizacion_lineas"]
sicetac_nucleo = tablas["piso-sicetac_nucleo"]
sicetac_raras = tablas["piso-sicetac_raras"]
tarifa_plaza = tablas["tarifa-plaza-tblSAPTarifas-2022-09-01_a_hoy"]

verificables = cotizacion_lineas[cotizacion_lineas["fuente"].isin(["Sicetac", "Tarifario"])].copy()
print(f"Líneas con Fuente en Sicetac/Tarifario: {len(verificables)}")

valores_sicetac = set(sicetac_nucleo["valorsiceconhora"].dropna()) | set(sicetac_raras["valorsiceconhora"].dropna())
valores_tarifario = set(tarifa_plaza["valor"].dropna())

def verificar(fila):
    if fila["fuente"] == "Sicetac":
        return fila["valorFlete"] in valores_sicetac
    if fila["fuente"] == "Tarifario":
        return fila["valorFlete"] in valores_tarifario
    return None

verificables["coincide_con_fuente_externa"] = verificables.apply(verificar, axis=1)

print(verificables[["id", "fuente", "valorFlete", "coincide_con_fuente_externa"]].to_string())
print()
print("Resumen por fuente:")
print(verificables.groupby("fuente")["coincide_con_fuente_externa"].agg(["sum", "count"]))

# La línea de la ruta malformada ya conocida (LineaCotizacionId CA0FC8A8...)
linea_malformada = verificables[verificables["id"] == "CA0FC8A8-6AB8-4206-9794-17DD283F2BD3"]
print("\nLínea de la ruta malformada (se espera que NO coincida):")
print(linea_malformada[["id", "fuente", "valorFlete", "coincide_con_fuente_externa"]].to_string())


Líneas con Fuente en Sicetac/Tarifario: 45
                                      id     fuente  valorFlete  coincide_con_fuente_externa
0   C5154B85-5774-43F8-8049-72B82F9AB805    Sicetac   4832788.0                        False
1   7828D2DD-1C4B-4E11-BBFE-A1ED81FFAE99    Sicetac   1802013.0                        False
2   B555A690-8131-4A86-999D-A827A5B094EE    Sicetac   8175231.0                        False
3   CA0FC8A8-6AB8-4206-9794-17DD283F2BD3    Sicetac   5783697.0                        False
4   337AA221-FD0A-4CF2-BE94-0038DE95D7EE  Tarifario   6400000.0                         True
5   3FB01715-E528-4450-A4D2-00D1C5382F66  Tarifario   2300000.0                         True
6   4C5F7605-92E8-442D-AEA4-11E495C6BFBE  Tarifario   8650000.0                         True
7   B316D6C3-5565-47B6-B19C-9F9CFC5850AA    Sicetac   1564425.0                        False
9   595073B5-346A-44B0-B1F4-A6E110AB1500  Tarifario   7667624.0                         True
10  5BC2911F-DC95-4608-9186

## Nota 🔎 — Verificación adicional de Sicetac contra la tabla viva (no reproducible desde este notebook)

Los 6 valores únicos de `Fuente=Sicetac` (excluyendo el malformado, ver
Paso 2) se buscaron también directamente contra `tblRutasSicetac` en la
base de datos operativa — no contra `piso-sicetac`, que es solo una
muestra de 151 filas. Esta consulta se corrió fuera de este notebook,
en el cliente de base de datos del usuario, así que **no es reproducible
desde una celda de código** — depende del estado de la base de datos en
el momento de la consulta, que cambia mes a mes.

```sql
SELECT VALORSICECONHORA, COUNT(*) AS veces_que_aparece
FROM tblRutasSicetac
WHERE VALORSICECONHORA IN (1564425, 1802013, 2153978, 2818633, 4832788, 8175231)
GROUP BY VALORSICECONHORA
ORDER BY VALORSICECONHORA;
```

**Resultado (2026-09-11):** solo `2153978` apareció (1 vez); los otros 5
no aparecieron. Consistente con que la tabla viva ya avanzó al período
`202609` (septiembre) y esas tarifas de agosto probablemente ya se
actualizaron — no con que el motor de costeo haya fallado en su momento.


## Paso 3 — Consistencia de las banderas booleanas (§5.4 del plan)

Varias columnas de sí/no vienen codificadas distinto según el lado
(`0`/`1` en lo propio, `"True"`/`"False"` en SAP, `"SI"`/`"NO"` en
`viaestandar`). Ya las convertimos a booleano real en el notebook 00 —
aquí confirmamos que esa conversión no dejó ningún valor vacío por
sorpresa (lo que pasaría si algún valor no coincidiera exactamente con
lo que esperábamos, ej. "true" en minúscula en vez de "True").


In [3]:
columnas_booleanas = {
    "catalogo_servicios_conexos": ["activo"],
    "catalogo_usuarios": ["activo"],
    "lineas_solicitud": ["exclusivo", "consolidado", "retornaContenedor", "levanteContenedorDiferente"],
    "tarifa-plaza-tblSAPTarifas-2022-09-01_a_hoy": ["devolucion", "consolidado", "exclusivo"],
    "historico-sap-tblSAPTarifasLog-2022-09-01_a_hoy": ["devolucion"],
    "piso-sicetac_nucleo": ["viaestandar"],
    "piso-sicetac_raras": ["viaestandar"],
}

print("Nulos inesperados en columnas booleanas (se espera 0 en todas):")
for nombre_tabla, columnas in columnas_booleanas.items():
    df = tablas[nombre_tabla]
    for col in columnas:
        nulos = df[col].isna().sum()
        print(f"  {nombre_tabla}.{col}: {nulos} nulos de {len(df)} filas")


Nulos inesperados en columnas booleanas (se espera 0 en todas):
  catalogo_servicios_conexos.activo: 0 nulos de 11 filas
  catalogo_usuarios.activo: 0 nulos de 11 filas
  lineas_solicitud.exclusivo: 0 nulos de 43 filas
  lineas_solicitud.consolidado: 0 nulos de 43 filas
  lineas_solicitud.retornaContenedor: 0 nulos de 43 filas
  lineas_solicitud.levanteContenedorDiferente: 0 nulos de 43 filas
  tarifa-plaza-tblSAPTarifas-2022-09-01_a_hoy.devolucion: 0 nulos de 23020 filas
  tarifa-plaza-tblSAPTarifas-2022-09-01_a_hoy.consolidado: 0 nulos de 23020 filas
  tarifa-plaza-tblSAPTarifas-2022-09-01_a_hoy.exclusivo: 0 nulos de 23020 filas
  historico-sap-tblSAPTarifasLog-2022-09-01_a_hoy.devolucion: 0 nulos de 25318 filas
  piso-sicetac_nucleo.viaestandar: 0 nulos de 126 filas
  piso-sicetac_raras.viaestandar: 0 nulos de 25 filas


## Síntesis del notebook 03 — Calidad del dato (Fase 4, §5 del plan)

**Objetivo cumplido:** cuantificar las 7 dimensiones de calidad del plan.
2 de las 7 (completitud, unicidad) ya estaban resueltas en notebooks
anteriores; este notebook cubrió validez y consistencia con código nuevo,
y declara el resto con lo ya sabido, sin necesidad de código adicional.

**Validez (Paso 1):** reconfirmados con código propio los 2 hallazgos
puntuales de HU-10.3 — 1 solicitud con `Deadline` anterior a su creación
(`3EAB0AB6-...`, sospechado también como error de captura manual del
usuario) y 3 rutas malformadas (`"222"/"222"`, `"b"/"j"`, `"tr"/"rr"` —
confirmadas como dato de prueba del usuario). Rangos físicos limpios: 0
casos fuera de 0-100 en `ProbabilidadCierre`/`ScorePrioridad`, 0 montos
negativos en las 4 tablas monetarias revisadas.

**Consistencia — Fuente=Sicetac/Tarifario (Paso 2):** reconfirmación
parcial del hallazgo más importante de HU-10.3.
- `Tarifario`: reconfirmado 100% (32/32) con código propio, contra el
  extracto casi completo de `tarifa-plaza` (23.020 filas).
- `Sicetac`: no reproducible con precisión — `piso-sicetac` es una
  muestra de solo 151 filas de ~2 millones, no diseñada para incluir las
  rutas de estas 13 solicitudes. Verificación adicional 🔎 (ver nota
  arriba, no reproducible desde una celda) contra la tabla viva de
  SICE-TAC encontró solo 1 de 6 valores — consistente con que las
  tarifas ya se actualizaron desde agosto, no con que el motor fallara.
  El hallazgo original de HU-10.3 (6/6 contra la foto de agosto) se
  mantiene como válido, sin poder re-verificarse hoy con precisión.

**Banderas booleanas (Paso 3):** 0 nulos inesperados en las 12 columnas
de sí/no revisadas — la conversión del notebook 00 quedó limpia.

**Declarado sin código nuevo (ya sabido, o corresponde a otra fase):**
- Exactitud (§5.5): se profundiza en la Fase 6 (bivariado) — es el
  corazón de la pregunta de negocio de este EDA.
- Integridad referencial (§5.6): cubierta en el notebook 02. Recordatorio
  heredado: la cuenta `consultascubos` tiene permisos de escritura
  indebidos en `DWCubosOperacionesEBS` (ya reportado a IT en HU-10.2/
  10.3, no bloquea este EDA).
- Cobertura temporal (§5.7): SICE-TAC solo tiene el período `202608` en
  el extracto (ya avanzó a `202609` en la base viva); lo propio cubre 10
  días de agosto-septiembre 2026; SAP (`tarifa-plaza`/`historico-sap`)
  tiene profundidad real desde 2024-01-31.

**Qué le queda listo al siguiente notebook:** las 7 dimensiones de
calidad cuantificadas, listas para consolidar en `reportes/hallazgos.md`
y `limitaciones.md`. El siguiente paso natural del plan es la Fase 5
(Univariado, §6) — la forma de cada variable, tabla por tabla.
